# Econ 3916 Final Project
## Predicting Income Class with the UCI Adult Dataset

**Akash Viswanathan**


## 1. Proposal

**Prediction question:** Can we predict whether an individual's annual income exceeds $50,000 using demographic and work-related census features?

**Prediction vs. causation:** This is a prediction task, not a causal analysis. The goal is to estimate whether someone belongs to the `>50K` or `<=50K` class based on observed features. A strong predictive relationship does not mean that any variable causes higher income.

**Dataset:** UCI Adult, also known as the Census Income dataset  
**Source URL:** https://archive.ics.uci.edu/dataset/2/adult

**Basic dataset stats used in this notebook:**
- Instances used in this analysis: 32,561 rows from `adult.data`
- Predictors: 14 demographic and employment-related features
- Target variable: `income`, coded as `>50K` or `<=50K`

**Stakeholder:** This analysis would help a workforce development organization decide which clients may be at higher risk of lower income outcomes, so it can target job training, advising, and support resources more effectively.

**Why this matters:** A predictive model can help a stakeholder prioritize outreach and identify broad labor-market patterns. The model should not be used as a final decision-maker because income is connected to sensitive social and economic factors, and the dataset contains demographic variables.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

## 2. Load the Data

In [2]:
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"
]

data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

df = pd.read_csv(
    data_url,
    header=None,
    names=columns,
    na_values="?",
    skipinitialspace=True
)

df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

Shape: (32561, 15)

Data types:
age                int64
workclass         object
fnlwgt             int64
education         object
education_num      int64
marital_status    object
occupation        object
relationship      object
race              object
sex               object
capital_gain       int64
capital_loss       int64
hours_per_week     int64
native_country    object
income            object
dtype: object


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       30725 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education_num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      30718 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital_gain    32561 non-null  int64 
 11  capital_loss    32561 non-null  int64 
 12  hours_per_week  32561 non-null  int64 
 13  native_country  31978 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


### Initial interpretation
The dataset contains a mix of numeric and categorical variables, which makes it a good fit for a supervised classification pipeline. The target variable is binary, so this will be a classification problem.

## 3. Data Quality and Missing Data Assessment

In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_percents = (df.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percents.round(2)
})

missing_summary[missing_summary["missing_count"] > 0]

### Missing-data framework (MCAR / MAR / MNAR)

The missing values appear mainly in categorical columns such as `workclass`, `occupation`, and `native_country`. In this dataset, missing values are represented by `?`, which are converted to `NaN` during loading.

I do not think the missingness is MCAR (Missing Completely at Random), because whether a work class or occupation is missing could plausibly depend on employment status, reporting issues, or other observed characteristics. It is more reasonable to treat this as MAR (Missing At Random), where missingness may depend on other observed variables.

**Chosen strategy:** For the checkpoint, I will keep the rows and handle missing values using imputation inside the machine learning pipeline:
- numeric features: median imputation
- categorical features: most-frequent imputation

This is a practical choice because it preserves sample size and avoids dropping many observations.

### Data quality summary

- The dataset has both numeric and categorical predictors.
- Missing values exist in a few categorical columns and are handled through imputation.
- No duplicate-row handling is applied at this stage because the dataset is already commonly used in benchmark form.
- The target variable is already labeled and suitable for binary classification.

## 4. Exploratory Data Analysis

In [ ]:
income_counts = df["income"].value_counts()

plt.figure(figsize=(6, 4))
income_counts.plot(kind="bar")
plt.title("Income Class Distribution")
plt.xlabel("Income")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

income_counts

**Interpretation:** The target classes are somewhat imbalanced, with more individuals in the `<=50K` category than in the `>50K` category. This matters because a model could appear accurate by overpredicting the majority class, so precision and recall are important metrics in addition to accuracy.

In [ ]:
plt.figure(figsize=(7, 4))
df.boxplot(column="age", by="income", grid=False)
plt.title("Age Distribution by Income Class")
plt.suptitle("")
plt.xlabel("Income")
plt.ylabel("Age")
plt.show()

**Interpretation:** Individuals in the >50K group appear older on average than those in the <=50K group. This suggests that age may carry useful predictive signal, possibly reflecting experience or career stage.

In [ ]:
edu_income = (
    df.groupby("education")["income"]
    .apply(lambda s: (s == ">50K").mean())
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 5))
edu_income.plot(kind="bar")
plt.title("Share of Individuals Earning >50K by Education Level")
plt.xlabel("Education")
plt.ylabel("Proportion with Income >50K")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

edu_income.head(10)

**Interpretation:** Higher education categories generally have a larger share of individuals earning more than $50K. This does not imply causation, but it does indicate that education level is likely to be an important predictor.

In [ ]:
hours_by_income = [
    df.loc[df["income"] == "<=50K", "hours_per_week"].dropna(),
    df.loc[df["income"] == ">50K", "hours_per_week"].dropna()
]

plt.figure(figsize=(7, 4))
plt.hist(hours_by_income, bins=25, label=["<=50K", ">50K"], alpha=0.8)
plt.title("Hours Worked per Week by Income Class")
plt.xlabel("Hours per week")
plt.ylabel("Count")
plt.legend()
plt.show()

**Interpretation:** The >50K group tends to be more concentrated around full-time and above-full-time work hours. This suggests that hours worked per week may also contribute predictive information.

## 5. Prepare Features and Target

In [ ]:
X = df.drop(columns="income")
y = df["income"]

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:", categorical_features)
print("\nNumeric features:", numeric_features)

## 6. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

## 7. Preliminary Model
### Logistic Regression Pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, solver="liblinear", random_state=42))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=">50K")
recall = recall_score(y_test, y_pred, pos_label=">50K")

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=["<=50K", ">50K"])
cm_df = pd.DataFrame(
    cm,
    index=["Actual <=50K", "Actual >50K"],
    columns=["Predicted <=50K", "Predicted >50K"]
)
cm_df

In [ ]:
print(classification_report(y_test, y_pred))

### Preliminary model interpretation

The logistic regression model is the first real benchmark for this income classification task. It is useful because it is relatively interpretable and gives a clear baseline before testing more flexible models. Since the classes are imbalanced, accuracy alone is not enough. Precision and recall for the `>50K` class are especially important because the stakeholder would care about correctly identifying people who may have higher or lower predicted income outcomes. This first model is not the final answer, but it provides a baseline for comparison against tree-based models below.

## 8. Final Project Extension: Model Comparison

For the final project, I compare several supervised classification models rather than relying on one preliminary model. I include a simple dummy baseline to show the minimum standard a model must beat, then compare logistic regression, a decision tree, a random forest, and gradient boosting.

The main evaluation focus is the `>50K` class because it is the minority class. I report accuracy, precision, recall, F1-score, and ROC AUC.

In [ ]:
final_models = {
    "Dummy baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, solver="liblinear", random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=50,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

model_results = []
fitted_models = {}

for model_name, clf in final_models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", clf)
    ])

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    fitted_models[model_name] = pipeline

    result = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision (>50K)": precision_score(y_test, preds, pos_label=">50K", zero_division=0),
        "Recall (>50K)": recall_score(y_test, preds, pos_label=">50K", zero_division=0),
        "F1 (>50K)": f1_score(y_test, preds, pos_label=">50K", zero_division=0)
    }

    if hasattr(pipeline.named_steps["classifier"], "predict_proba"):
        probs = pipeline.predict_proba(X_test)
        positive_index = list(pipeline.named_steps["classifier"].classes_).index(">50K")
        result["ROC AUC"] = roc_auc_score((y_test == ">50K").astype(int), probs[:, positive_index])
    else:
        result["ROC AUC"] = np.nan

    model_results.append(result)

metrics_df = pd.DataFrame(model_results).sort_values("F1 (>50K)", ascending=False)
metrics_df.round(4)

### Model comparison results

The model comparison from my run produced the following results:

| Model | Accuracy | Precision (>50K) | Recall (>50K) | F1 (>50K) | ROC AUC |
|---|---:|---:|---:|---:|---:|
| Gradient Boosting | 0.8689 | 0.7898 | 0.6205 | 0.6950 | 0.9234 |
| Decision Tree | 0.8578 | 0.7248 | 0.6601 | 0.6909 | 0.9071 |
| Logistic Regression | 0.8554 | 0.7393 | 0.6167 | 0.6725 | 0.9080 |
| Random Forest | 0.8610 | 0.8000 | 0.5638 | 0.6614 | 0.9152 |
| Dummy baseline | 0.7593 | 0.0000 | 0.0000 | 0.0000 | 0.5000 |

Gradient boosting is the best overall model by F1-score for the `>50K` class and also has the highest ROC AUC. The decision tree has slightly higher recall than gradient boosting, but gradient boosting has better precision, better F1-score, and better overall ranking ability. For this reason, I select gradient boosting as the final model.

In [ ]:
best_model_name = "Gradient Boosting"
best_model = fitted_models[best_model_name]
best_preds = best_model.predict(X_test)

best_cm = confusion_matrix(y_test, best_preds, labels=["<=50K", ">50K"])
best_cm_df = pd.DataFrame(
    best_cm,
    index=["Actual <=50K", "Actual >50K"],
    columns=["Predicted <=50K", "Predicted >50K"]
)

best_cm_df

In [ ]:
print(classification_report(y_test, best_preds))

### Final model interpretation

The selected gradient boosting model reached 0.8689 accuracy, 0.7898 precision for the `>50K` class, 0.6205 recall for the `>50K` class, and 0.9234 ROC AUC. This is meaningfully better than the dummy baseline and slightly stronger than the other real models when balancing precision and recall.

The confusion matrix shows that the model is especially strong at identifying the majority `<=50K` group. It still misses some `>50K` cases, which is expected because the target is imbalanced and the higher-income group is harder to identify. For the stakeholder, this means the model is useful as a screening or prioritization tool, but it should not be used as an automatic decision system.

## 9. Feature Importance

For the final model, I also examine feature importance. Since gradient boosting uses the transformed feature matrix, the code below reconstructs the feature names after one-hot encoding and shows the most influential model features.

In [ ]:
# Extract transformed feature names
ohe = best_model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
categorical_feature_names = ohe.get_feature_names_out(categorical_features)
all_feature_names = np.concatenate([numeric_features, categorical_feature_names])

importances = best_model.named_steps["classifier"].feature_importances_

importance_df = (
    pd.DataFrame({
        "feature": all_feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
    .head(15)
)

importance_df

In [ ]:
plt.figure(figsize=(9, 5))
plt.barh(importance_df["feature"][::-1], importance_df["importance"][::-1])
plt.title("Top 15 Feature Importances: Gradient Boosting")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

### Feature importance interpretation

The most important variables are typically related to capital gains, marital or relationship status, education, age, and hours worked per week. This result is consistent with the EDA: income class is not explained by one variable alone, but by a combination of employment, education, demographic, and financial features. These are predictive patterns only, not proof that any variable causes higher income.

## 10. Deployment Plan: Streamlit Dashboard

The final deliverable includes a Streamlit dashboard file named `streamlit_app.py`. The dashboard is designed for Streamlit Community Cloud and includes:

- project overview and stakeholder framing
- dataset preview
- target distribution
- model comparison table
- final model confusion matrix
- individual prediction tool

The dashboard should be deployed by pushing the project folder to GitHub and connecting the repository to Streamlit Community Cloud. The main file path should be set to `streamlit_app.py`.

In [ ]:
# Optional artifact export for the final repository
metrics_df.round(4).to_csv("model_metrics.csv", index=False)
importance_df.to_csv("feature_importances.csv", index=False)

print("Saved model_metrics.csv and feature_importances.csv")

## Final Conclusion

This project shows that census-based demographic and employment features can predict income class with meaningful accuracy, but the model is imperfect and should be interpreted carefully. Gradient boosting performed best overall, with the strongest F1-score for the `>50K` class and the highest ROC AUC among the models tested.

The final recommendation is to use the model as a decision-support tool for broad workforce-planning or outreach prioritization, not as a final decision-maker. The model can help identify patterns and allocate attention, but human review is still necessary because the dataset includes demographic variables and reflects real-world social inequality.